# Incident Timeline Fusion

Combine synthetic public reports into reliability-weighted incident time windows.

**Safety and scope:** This notebook uses deterministic synthetic data and makes no network requests. Its results are analytical leads, not attribution or identity claims.

## Goal

Create an auditable timeline that highlights corroborated events and visible contradictions.


## Setup

The workflow runs offline with NumPy and Pandas. Parameters and source-like fields are visible so the analysis can be reviewed and rerun.

### Key Assumptions

- All records are synthetic and contain no real people or infrastructure.
- Scores prioritize review; they do not prove ownership, intent, identity, or location.
- Real use requires documented authority, provenance, source terms, and retention limits.


In [1]:
import numpy as np
import pandas as pd

SEED = 88
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
pd.set_option("display.max_colwidth", 70)


## Steps

### 1. Create bounded synthetic observations


In [2]:
observation_count = 160
source_types = np.array(["official", "journalism", "technical-report", "community"])
source_reliability = {"official": 0.90, "journalism": 0.78, "technical-report": 0.86, "community": 0.48}
observations = pd.DataFrame({
    "observation_id": [f"obs-{index:04d}" for index in range(observation_count)],
    "source_type": rng.choice(source_types, observation_count, p=[0.18, 0.32, 0.28, 0.22]),
    "minute": rng.choice([30, 90, 150, 240, 360], observation_count, p=[0.12, 0.24, 0.30, 0.22, 0.12]) + rng.integers(-18, 19, observation_count),
    "event_type": rng.choice(["initial-access", "service-impact", "containment", "recovery"], observation_count),
    "contradiction": rng.binomial(1, 0.10, observation_count),
})
observations["reliability"] = observations["source_type"].map(source_reliability)
observations["time_window"] = (observations["minute"] // 30) * 30
print(observations.head(8).to_string(index=False))


observation_id      source_type  minute     event_type  contradiction  reliability  time_window
      obs-0000       journalism     356       recovery              1         0.78          330
      obs-0001 technical-report      40    containment              0         0.86           30
      obs-0002 technical-report     132       recovery              0         0.86          120
      obs-0003 technical-report     352 initial-access              0         0.86          330
      obs-0004 technical-report      40       recovery              0         0.86           30
      obs-0005         official     152 initial-access              0         0.90          150
      obs-0006         official     103    containment              0         0.90           90
      obs-0007       journalism     102    containment              1         0.78           90


### 2. Analyze and rank the observations


In [3]:
observations["evidence_weight"] = observations["reliability"] * (1 - 0.55 * observations["contradiction"])
timeline = observations.groupby(["time_window", "event_type"], as_index=False).agg(
    reports=("observation_id", "count"),
    source_types=("source_type", "nunique"),
    evidence_weight=("evidence_weight", "sum"),
    contradictions=("contradiction", "sum"),
)
timeline["corroboration_score"] = (
    0.45 * np.minimum(timeline["evidence_weight"] / 8, 1)
    + 0.35 * np.minimum(timeline["source_types"] / 4, 1)
    + 0.20 * (1 - np.minimum(timeline["contradictions"] / timeline["reports"], 1))
).round(3)
ranked_events = timeline.sort_values("corroboration_score", ascending=False)
print(ranked_events.head(10).to_string(index=False))


 time_window     event_type  reports  source_types  evidence_weight  contradictions  corroboration_score
         210    containment        8             4            6.460               0                0.913
         120       recovery       10             3            7.867               1                0.885
         150    containment        9             3            7.300               0                0.873
         150       recovery        7             4            4.760               0                0.818
         120    containment        5             4            3.800               0                0.764
          90    containment        7             4            4.636               2                0.754
         150 service-impact        6             4            3.905               1                0.736
         120 service-impact        6             3            4.240               0                0.701
         120 initial-access        5             3     

## Checks

Run deterministic integrity and reasonableness checks.


In [4]:
assert observations["observation_id"].is_unique
assert ranked_events["corroboration_score"].between(0, 1).all()
assert ranked_events["reports"].sum() == observation_count
print("Checks passed; contradictions remain visible instead of being silently discarded.")


Checks passed; contradictions remain visible instead of being silently discarded.


## Next Steps

- Attach citations and archived source timestamps to every observation.
- Separate reported time, publication time, and analyst-inferred time.
